In [1]:
import torch
import cying.nn as cynn
import matplotlib.pyplot as plt
import numpy as np

In [2]:
def bytes_to_unicode():
    # 原始保留的字节范围
    bs = (
        list(range(ord("!"), ord("~") + 1)) +          # ASCII可打印字符（33-126）
        list(range(ord("¡"), ord("¬") + 1)) +          # 西班牙语特殊字符（161-172）
        list(range(ord("®"), ord("ÿ") + 1))            # 其他扩展字符（174-255）
    )
    
    cs = bs.copy()  # 初始字符列表
    n = 0
    
    # 遍历所有可能的字节（0-255）
    for b in range(2**8):
        if b not in bs:
            bs.append(b)
            cs.append(2**8 + n)  # 超出原始范围的字节映射到更高Unicode码位
            n += 1
    
    # 将码位转换为Unicode字符
    cs = [chr(code) for code in cs]
    
    return dict(zip(bs, cs))

def get_reverse_mapping(forward_map):
    """
    根据正向映射生成反向映射
    返回字典：{unicode_char: byte_value}
    """
    return {v: k for k, v in forward_map.items()}

# 生成映射表
forward_map = bytes_to_unicode()
reverse_map = get_reverse_mapping(forward_map)
from tokenizers import Tokenizer
tokenizer = Tokenizer.from_file("./cying/datasets/tokenizer-wiki.json")
def unicode_str_to_bytes(unicode_str):
    return bytes([reverse_map[c] for c in unicode_str])

In [15]:
device = torch.device('cuda:0')
opt_lang_model = torch.load('./models/model.pth',map_location=device,weights_only=False)

In [16]:
token_seq = '今天是美好的一天，我们从早上十点半开始，在兰州大学榆中校区上讨论班。早上的讨论班持续到中午12点30分，然后去吃饭，吃到1点半以后，休息了半个小时。下午的讨论班在两点半开始，一直持续到六点，然后晚上七点开始继续上，目前还没有结束。今天的讨论班本安排在周日，由于邓老师出差了，所以调整到了今天（也就是周一）。'
# token_seq = 'Today is a good day, do you want to go play?'
token_seq_encode = tokenizer.encode(token_seq).ids
len_ = len(token_seq_encode) + 2
token_seq_encode = torch.tensor([0] + token_seq_encode + [1,2]).unsqueeze(0)
word = []
while True:
    word.append(opt_lang_model(
        token_seq_encode.to(device),
        casual_mask=torch.triu(torch.ones(token_seq_encode.shape[1], token_seq_encode.shape[1]), diagonal=1).bool().to(device)
    )[0,-1].argmax().cpu())
    if word[-1] == 3:
        break
    else:
        token_seq_encode = torch.cat([token_seq_encode,word[-1][None,None]],dim=1)
        unicode_word = tokenizer.decode(word).replace(' ','')
        try:
           recovered_word = unicode_str_to_bytes(unicode_word).decode('utf-8')
        except UnicodeDecodeError:
            print('',end='')
        else:
            print(recovered_word,end='')
        finally:
            word=[]

Today is a good day, we started to discuss class in the morning and semi-school in the morning of the University of Lanzhou.The following discussion on the break-up to the middle of the morning 12:30 p.m. and then the rest of the rest is half an hour.The next meeting of the class is on the other half of the two or more, and the evening is about six or more, and the end of the evening is not over.Today's discussion class is on Sunday, because the teacher is out of the day, so adjusted to today (that is the week).

In [97]:
token_seq_encode.shape

torch.Size([1, 399])

In [ ]:
import torch
import cying.nn as cynn
device = torch.device('cpu')

opt_lang_model = torch.load('./models/model7.pth', map_location=device, weights_only=False)

import json
from tokenizers import Tokenizer
from torch.utils.data import Dataset, DataLoader

scale = 2
max_seq_len = 512
batch_size = 64 // scale

class Trans2019(Dataset):
    def __init__(self, filepath, tokenizer, seq_len=64):
        self.tokenizer = tokenizer
        self.seq_len = seq_len
        self.data_english = []
        self.data_chinese = []
        with open(filepath, "r", encoding="utf-8") as f:
            for line in f:
                datai = json.loads(line)
                self.data_english.append(datai['english'])
                self.data_chinese.append(datai['chinese'])
    
    def __getitem__(self, index):
        text_english = self.data_english[index]
        text_chinese = self.data_chinese[index]
        seq_len = len(text_english) + len(text_chinese)
        cut_rand = torch.rand(1)
        while seq_len/self.seq_len < cut_rand:
            index_rand = torch.randint(0,self.__len__(),(1,))[0]
            text_english += self.data_english[index_rand]
            text_chinese += self.data_chinese[index_rand]
            seq_len = len(text_english) + len(text_chinese)
        if torch.randn(1) < 0:
            tokens_english = [0] + self.tokenizer.encode(text_english).ids + [1]
            tokens_chinese = [2] + self.tokenizer.encode(text_chinese).ids + [3]
            token_seq = tokens_english + tokens_chinese[:-1]
            tgt_seq = [4]*len(tokens_english) + tokens_chinese[1:]
        else:
            tokens_english = [2] + self.tokenizer.encode(text_english).ids + [3]
            tokens_chinese = [0] + self.tokenizer.encode(text_chinese).ids + [1]
            token_seq = tokens_chinese + tokens_english[:-1]
            tgt_seq = [4]*len(tokens_chinese) + tokens_english[1:]
        if len(token_seq) >= self.seq_len:
            token_seq = token_seq[:self.seq_len]
            tgt_seq = tgt_seq[:self.seq_len]
        else:
            token_seq = token_seq + [4]*(self.seq_len - len(token_seq))
            tgt_seq = tgt_seq + [4]*(self.seq_len - len(tgt_seq))
        input_ids = torch.tensor(token_seq, dtype=torch.long)
        target_ids = torch.tensor(tgt_seq, dtype=torch.long)
        return input_ids, target_ids

    def __len__(self):
        return len(self.data_english) // batch_size * batch_size

tokenizer = Tokenizer.from_file("./cying/datasets/tokenizer-wiki.json")
valid_dataset = Trans2019(
    filepath=f'./cying/datasets/translation2019zh/translation2019zh_valid.json',
    tokenizer=tokenizer,
    seq_len=max_seq_len
)
valid_loader = DataLoader(
    dataset=valid_dataset,
    batch_size=batch_size
)

from tqdm import tqdm
epoch = 10**2
criterion = torch.nn.CrossEntropyLoss(ignore_index=4)

with torch.no_grad():
    opt_lang_model.eval()
    pbar = tqdm(valid_loader)
    for token_ids, tgt_ids in pbar:
        token_ids = token_ids.to(device)
        tgt_ids = tgt_ids.to(device)
        outputs = opt_lang_model(
            token_seq=token_ids,
            casual_mask=torch.triu(torch.ones(max_seq_len, max_seq_len), diagonal=1).bool().to(device)
        )
        loss = criterion(
            outputs.view(-1, outputs.size(-1)),
            tgt_ids.reshape(-1)
        )
        pbar.set_description(f"Loss {loss.item():.4e}")